# Metrics Analysis

This notebook demonstrates how to analyse metrics exported by the observability stack's DuckDB export CLI.

The export tool writes metrics into a DuckDB database file with the following schema:

```sql
CREATE TABLE metrics (
    timestamp       TIMESTAMP NOT NULL,
    metric_name     VARCHAR NOT NULL,
    labels          MAP(VARCHAR, VARCHAR),
    value           DOUBLE NOT NULL,
    export_time     TIMESTAMP
);
```

DuckDB's SQL dialect provides powerful analytical functions that make it straightforward to
explore time-series metrics data without standing up a full metrics backend.

In [ ]:
# Install duckdb if not already available
%pip install --quiet duckdb

import duckdb

# Connect to the exported DuckDB file.
# Replace the path below with the actual exported database file.
# The export CLI typically writes files named like:
#   telemetry-2026-02-14T12-00-00.duckdb
#
# List available files with:  !ls ../../observability-export/telemetry-*.duckdb

DB_PATH = "../../observability-export/telemetry-export.duckdb"  # <-- update this path

con = duckdb.connect(DB_PATH, read_only=True)
print(f"Connected to {DB_PATH}")
print(f"Tables: {[row[0] for row in con.execute('SHOW TABLES').fetchall()]}")

In [ ]:
# Top 20 metrics by sample count
#
# Quick overview of which metrics have the most data points,
# helping identify the most actively scraped or emitted metrics.

con.sql("""
    SELECT
        metric_name,
        COUNT(*) AS sample_count
    FROM metrics
    GROUP BY metric_name
    ORDER BY sample_count DESC
    LIMIT 20
""").show()

In [ ]:
# Metric statistics: average, min, max, and time range per metric
#
# Useful for understanding the data coverage period and value distributions.

con.sql("""
    SELECT
        metric_name,
        MIN(timestamp)  AS first_seen,
        MAX(timestamp)  AS last_seen,
        AVG(value)      AS avg_value,
        MIN(value)      AS min_value,
        MAX(value)      AS max_value
    FROM metrics
    GROUP BY metric_name
    ORDER BY metric_name
""").show()

In [ ]:
# Time-range filtering
#
# Filter metrics to a specific time window and metric name.
# Adjust the date range and metric_name to match your data.

con.sql("""
    SELECT
        timestamp,
        metric_name,
        labels,
        value
    FROM metrics
    WHERE timestamp BETWEEN '2026-01-01' AND '2026-12-31'
      AND metric_name = 'up'
    ORDER BY timestamp
""").show()